# ChemView — Interactive Trajectory Analysis

`ChemView` is a wrapper around [Chemiscope](https://chemiscope.org/) that turns
ASE trajectories into linked scatter-plot + 3-D structure viewers inside Jupyter
notebooks. You define which properties to compute via short string *specs*;
ChemView handles the rest.

This tutorial uses a **glycine AIMD trajectory** (3 624 frames, 10 atoms/frame)
computed at the DFT level. The examples progress from basic energy monitoring to
conformational analysis, bond correlations, and atom-level force inspection.

---

## Glycine atom indices

| Index | Symbol | Role |
|------:|:------:|:-----|
| 0 | N | Amino nitrogen |
| 1 | C | Cα (alpha carbon) |
| 2 | C | Carboxyl carbon |
| 3 | O | Carbonyl oxygen (C=O) |
| 4 | O | Hydroxyl oxygen (C–OH) |
| 5 | H | Hα₁ |
| 6 | H | Hα₂ |
| 7 | H | HN₁ |
| 8 | H | HN₂ |
| 9 | H | H(OH) |

---

> **Static viewers:** Interactive panels below use pre-generated data.
> In your own Jupyter session `ChemView(...)` returns a live widget linked to your kernel.

In [ ]:
from ase.io import read
from sparc.src.utils.chemview import ChemView

## Load the trajectory

Any ASE-readable format works (`.traj`, `.xyz`, `.extxyz`, VASP `OUTCAR`, …).

In [ ]:
traj = read('_static/TrajCombined.traj', index=':')
print(f"Frames: {len(traj)},  atoms/frame: {len(traj[0])},  "
      f"symbols: {traj[0].get_chemical_symbols()}")

Frames: 3624,  atoms/frame: 10,  symbols: ['N', 'C', 'C', 'O', 'O', 'H', 'H', 'H', 'H', 'H']


---

## Example 1 — Energy landscape

The simplest usage: plot **potential energy vs frame index** to monitor
thermalisation, drift, or phase transitions over the simulation.

`map_color` sets the scatter-plot colour property — here energy itself gives
a continuous colour gradient from low (blue) to high (red).

In [ ]:
ChemView(
    frames=traj,
    specs=["frame", "energy"],
    names=["frame", "energy"],
    x="frame",
    y="energy",
    map_color="energy",
    meta_name="Glycine AIMD — Energy landscape",
)

ChemiscopeWidget(meta={'name': 'ChemView'}, settings={'target': 'structure', 'structure': [{'unitCell': False…

---

## Example 2 — Carbonyl and hydroxyl bond dynamics

Track the **C=O** (atoms 2→3) and **C–OH** (atoms 2→4) bond lengths
simultaneously. Scatter-plot axes are frame vs C=O distance, coloured by
energy. Clicking a point in the scatter plot highlights the corresponding
frame in the 3-D viewer.

When the same spec type appears more than once, use `names` to assign
distinct, human-readable keys.

In [ ]:
ChemView(
    frames=traj,
    specs=["frame", "energy", "distance:2,3", "distance:2,4"],
    names=["frame", "energy", "d_CO", "d_COH"],
    x="frame",
    y="d_CO",
    map_color="energy",
    meta_name="Glycine AIMD — Carboxyl bond dynamics",
)

ChemiscopeWidget(meta={'name': 'ChemView'}, settings={'target': 'structure', 'structure': [{'unitCell': False…

---

## Example 3 — Conformational landscape (Ramachandran-style)

Map the **N–Cα–C angle** (atoms 0,1,2) against the **N–Cα–C–O dihedral**
(atoms 0,1,2,3) — a Ramachandran-style projection that reveals which
conformations are sampled during the AIMD run.

Each scatter-plot point is one MD frame. Clustering or gaps in this plot
indicate preferred conformations or energy barriers.

In [ ]:
ChemView(
    frames=traj,
    specs=["energy", "angle:0,1,2", "dihedral:0,1,2,3"],
    names=["energy", "ang_NCaC", "phi"],
    x="ang_NCaC",
    y="phi",
    map_color="energy",
    meta_name="Glycine AIMD — Conformational landscape",
)

ChemiscopeWidget(meta={'name': 'ChemView'}, settings={'target': 'structure', 'structure': [{'unitCell': False…

---

## Example 4 — Bond-length correlations

Plot **C–N** vs **C=O** bond lengths to expose their dynamical correlation.
Resonance effects in the carboxyl group couple these two bonds: elongation
of one tends to shorten the other. Colour by **C–OH** bond length to add a
third dimension.

All four bond distances are computed in a single `ChemView` call; the
axes and colour can be changed interactively in the viewer.

In [ ]:
ChemView(
    frames=traj,
    specs=["frame", "energy",
           "distance:0,1",   # N–Cα
           "distance:1,2",   # Cα–C
           "distance:2,3",   # C=O
           "distance:2,4"],  # C–OH
    names=["frame", "energy", "d_NCa", "d_CaC", "d_CO", "d_COH"],
    x="d_NCa",
    y="d_CO",
    map_color="d_COH",
    meta_name="Glycine AIMD — Bond-length correlations",
)

ChemiscopeWidget(meta={'name': 'ChemView'}, settings={'target': 'structure', 'structure': [{'unitCell': False…

---

## Example 5 — Atom-level forces and index colouring

Atom-level specs (one value per atom per frame) expose per-site dynamics.
Here `force_norm` stores the DFT force magnitude |**F**ᵢ| for every atom
in every frame.

* `color_atoms="atom_index"` colours atoms by their index (rainbow scale)
  so you can visually track individual atoms across frames.
* `show_index=True` displays the integer index next to each atom in the
  3-D viewer — useful when building new specs.

Click any atom in the structure panel to see its `force_norm`, `atom_index`,
and any other per-atom properties in the info panel.

In [ ]:
ChemView(
    frames=traj,
    specs=["frame", "energy", "force_norm"],
    names=["frame", "energy", "force_norm"],
    x="frame",
    y="energy",
    map_color="energy",
    color_atoms="atom_index",
    show_index=True,
    meta_name="Glycine AIMD — Atom forces",
)

ChemiscopeWidget(meta={'name': 'ChemView'}, settings={'target': 'structure', 'structure': [{'unitCell': False…

---

## Example 6 — Single-structure viewer

A single `ase.Atoms` object can be passed directly — useful for inspecting
an optimised geometry or the lowest-energy frame of a trajectory.

Set `plot=False` to suppress the scatter plot and show only the 3-D viewer.
Atom-level position specs (`x_pos`, `y_pos`, `z_pos`) let you colour atoms
by their Cartesian coordinates.

In [ ]:
import numpy as np

# pick the minimum-energy frame
energies = [float(np.asarray(at.get_potential_energy()).flat[0]) for at in traj]
min_frame = traj[int(np.argmin(energies))]

ChemView(
    frames=[min_frame],
    specs=["x_pos", "y_pos", "z_pos"],
    names=["x", "y", "z"],
    x="x",
    y="y",
    plot=False,
    color_atoms="atom_index",
    show_index=True,
    meta_name="Glycine — lowest-energy frame",
)

StructureWidget(meta={'name': 'ChemView'}, settings={'target': 'structure', 'structure': [{'unitCell': False,…

---

## Tips and reference

### Supported specs

| Spec | Target | Unit | Notes |
|:-----|:------:|:----:|:------|
| `"frame"` | structure | — | Frame index |
| `"energy"` | structure | eV | From `atoms.get_potential_energy()` |
| `"distance:i,j"` | structure | Å | Bond/non-bond distance |
| `"angle:i,j,k"` | structure | deg | Valence angle at j |
| `"dihedral:i,j,k,l"` | structure | deg | Torsion angle |
| `"volume"` | structure | Å³ | Periodic cell volume |
| `"cell_a"`, `"cell_b"`, `"cell_c"` | structure | Å | Lattice lengths |
| `"cell_alpha"`, `"cell_beta"`, `"cell_gamma"` | structure | deg | Lattice angles |
| `"x_pos"`, `"y_pos"`, `"z_pos"` | atom | Å | Cartesian positions |
| `"force_norm"` | atom | eV/Å | Force magnitude |
| `"force_x"`, `"force_y"`, `"force_z"` | atom | eV/Å | Force components |
| `"symbol"` | atom | — | Element symbol |
| `"atomic_number"` | atom | — | Atomic number Z |

`atom_index` is always added automatically — no need to list it in `specs`.

### Key parameters

```python
ChemView(
    frames=traj,           # list of ase.Atoms
    specs=[...],           # property specs to compute
    names=[...],           # optional: rename keys (same length as specs)
    x="frame",             # scatter-plot x axis
    y="energy",            # scatter-plot y axis
    z=None,                # optional: 3-D scatter z axis
    map_color=None,        # scatter-plot colour property
    color_atoms="element", # 3-D viewer colouring: "element" | "atom_index" | None
    labels=False,          # show element labels on atoms
    show_index=False,      # show integer index on atoms
    env_cutoff=3.5,        # environment sphere radius (Å)
    plot=True,             # show scatter-plot panel
    meta_name="ChemView",  # dataset label
)
```